***Alexandre Mathias DONNAT, Sr - Télécom Paris***

### === Purpose ===
The goal of this lab is to disambiguate entities in a text. For example, given a Wikipedia article:

    Paris #17
    <Paris> is a figure in the Greek mythology.

the goal is to determine that Paris #17 = yago:Paris_(mythology).
Here, Paris #17 is an artificial title of the Wikipedia article, and yago:Paris_(mythology) is the unambiguous entity in the YAGO knowledge base. (https://yago-knowledge.org/resource/Elvis_Presley?search=Paris)

### === Provided Data ===

We provide
1. a preprocessed version of Wikipedia, wikipedia-corpus.txt, which contains ambiguous article titles with their content, as above.
2. a simplified version of the YAGO knowledge base, yago-sample.tsv.
4. a gold standard sample, student-gold-standard.tsv.

### === Task ===

Our task is to complete the function disambiguate() in this file.
It receives as input (1) the ambiguous Wikipedia title ("Paris" in the example), and (2) the article content, with the first occurence of the title entity signaled with "<"+entity+">".
The method shall return the unambiguous entity from YAGO.
The lab will be graded by a variant of the F1 score that gives higher weight to precision (with beta=0.5).


### === Working with Colab ===
We need to save a local copy of the notebook to our own google drive.
Connect to an execution environment using a GPU (this should be automatic, but be aware of this !). Upload the local files directly to the colab, and we can run everything !

Don't forget to download the results file at the end.

In [2]:
"""
Install necessary modules, run only once !
"""
!pip install -q transformers
!pip install -q sentencepiece
!pip install -q accelerate

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
from collections import defaultdict
import time
import time
import re
from collections import defaultdict

c:\Users\amdpr\OneDrive\ISK 2025\Escola\Telecom 2026\Cours\Period 4\NLP\Disambugation_Lab\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
"""
Loads a T5 LLM
"""
torch.cuda.empty_cache()
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large", device_map="auto")

In [1]:
class WikipediaArticle:
  """ Represents a Wikipedia article. Do not modify. """
  def __init__(self, title, content):
    self.title=title
    self.name=title[0:title.find("#")-1]
    self.entity_id = title[title.find("#")+1:]
    self.content=content

def wikipediaArticles(file):
  """ Yields the Wikipedia articles from a file. Do not modify. """
  article=[]
  title=None
  with open(file, "rt", encoding="utf=8") as inputFile:
    for line in inputFile:
      line=line.rstrip()
      if not title:
        title=line
        continue
      if not len(line) and title and len(article):
        yield WikipediaArticle(title, article[0])
        title=None
        article=[]
        continue
      article+=[line]

def clean(yagoEntity):
    """ Removes prefixes etc."""
    if yagoEntity.startswith('"'):
      return yagoEntity[1:-1]
    yagoEntity=yagoEntity[yagoEntity.find(':')+1:]
    return '<'+yagoEntity+'>'

In [ ]:
def run_evaluation():
  """Evaluation script, do not modify (unless we want to remove some prints).
  We use the f-05 measure, which gives more importance to precision: classifying entities correctly is more valued than finding all entities.
  """
  with open("student-gold-standard.tsv", "r", encoding="utf-8") as f:
    lines = f.readlines()
  gold_standard_dict = defaultdict(dict)
  for line in lines:
    title, entity_id, yagoentity = tuple(line.replace("\n","").split("\t"))
    gold_standard_dict[title][entity_id] = yagoentity
  gold_standard_dict = dict(gold_standard_dict)

  with open("results.tsv", "r", encoding="utf-8") as f:
    lines = f.readlines()
  predictions_dict = defaultdict(dict)
  for line in lines:
    title, entity_id, yagoentity = tuple(line.replace("\n","").split("\t"))
    predictions_dict[title][entity_id] = yagoentity

  true_pos = 0
  false_pos = 0
  false_neg = 0

  for title in predictions_dict:
    for entity_id in predictions_dict[title]:
      try:
        gold_yago_entity = gold_standard_dict[title][entity_id]
      except KeyError: #should not happen
        continue
      if predictions_dict[title][entity_id] == gold_yago_entity:
        true_pos += 1
      else:
        false_pos += 1
        if false_pos < 100:
          print("You disambiguated", title + " #" + entity_id, "wrong.", "Expected output: ", gold_yago_entity, ",given:", predictions_dict[title][entity_id])

  for gold_title in gold_standard_dict: #do we really want to measure this? There are some entities that don't have a wikipedia article, so they count. Should they be removed from the gold standard?
    for entity_id in gold_standard_dict[gold_title]:
      try:
        predict_entity = predictions_dict[gold_title][entity_id]
      except KeyError:
        false_neg += 1
        if false_neg < 100:
          print("You did not disambiguate", gold_title + " #" + entity_id + ".")

  if true_pos + false_pos != 0:
    precision = float(true_pos) / (true_pos + false_pos)
  else:
    precision = 0.0

  if true_pos + false_neg != 0:
    recall = float(true_pos) / (true_pos + false_neg)
  else:
    recall = 0.0

  beta = 0.5

  if precision + recall != 0.0:
    f05 = (1 + beta * beta) * precision * recall / (beta * beta * precision + recall)
  else:
    f05 = 0.0

  print()
  print("Scores (scaled from 0 to 100)")
  print("Precision", precision*100)
  print("Recall", recall*100)
  print("F-0.5 Score", f05*100)

In [3]:
class KnowledgeBase:
  """
  A simple knowledge base. Don't modify this code.

  Load the knowledge base:
      kb = KnowledgeBase("yago-sample.tsv")

  Access facts:
      spousesOfElvis = kb.facts["<Elvis_Presley>"]["<spouse>"]

  Access inverse facts:
      entitiesCalledParis = kb.inverseFacts['Paris']["<label>"]
  """
  __author__ = "Fabian Suchanek"

  def __init__(self, yagoFile):
    self.facts = {}
    self.inverseFacts = {}
    with open(yagoFile, encoding="utf-8") as file:
      print("Loading", yagoFile, end="...", flush=True)
      for line in file:
        split_line = line.split('\t')
        if len(split_line) < 3:
          raise RuntimeError("The file is not a valid KB file")
        subject = clean(split_line[0])
        relation = clean(split_line[1])
        obj = clean(split_line[2])
        self.facts.setdefault(subject, {})
        self.facts[subject].setdefault(relation, set())
        self.facts[subject][relation].add(obj)
        self.inverseFacts.setdefault(obj, {})
        self.inverseFacts[obj].setdefault(relation, set())
        self.inverseFacts[obj][relation].add(subject)
      print("done", flush=True)

In [ ]:
"""
Runs the language model with prompt (string) as input
"""
device = "cuda" if torch.cuda.is_available() else "cpu"
def run_model(prompt):
  input_text = prompt
  input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

  outputs = model.generate(input_ids, num_beams=1)
  return(tokenizer.decode(outputs[0]))

In [5]:
import re

STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "at", "to", "for", "from", "by", "with",
    "and", "or", "as", "is", "was", "are", "were", "be", "been", "this", "that",
    "it", "its", "he", "she", "they", "his", "her", "their", "also"
}

def tokenize(text):
    text = text.replace("<", " ").replace(">", " ")
    text = text.replace("_", " ")
    tokens = re.findall(r"[A-Za-z0-9]+", text.lower())
    return {t for t in tokens if len(t) > 2 and t not in STOPWORDS}

def entity_to_text(entity):
    """
    Converts <Paris_(mythology)> into readable text: Paris mythology
    """
    return entity.replace("<", "").replace(">", "").replace("_", " ")

def get_candidates(entityName, knowledgeBase):
    """
    Finds YAGO entities whose label matches the ambiguous Wikipedia title.
    Example: "Paris" -> {<Paris>, <Paris_(mythology)>, ...}
    """
    candidates = set()

    # Exact label match
    if entityName in knowledgeBase.inverseFacts:
        candidates |= knowledgeBase.inverseFacts[entityName].get("<label>", set())

    # Case-insensitive fallback
    if not candidates:
        target = entityName.lower()
        for label, relations in knowledgeBase.inverseFacts.items():
            if isinstance(label, str) and label.lower() == target:
                candidates |= relations.get("<label>", set())

    return candidates

def candidate_context(candidate, knowledgeBase):
    """
    Builds a text description of a candidate using all its YAGO facts.
    """
    parts = [entity_to_text(candidate)]

    if candidate in knowledgeBase.facts:
        for relation, objects in knowledgeBase.facts[candidate].items():
            parts.append(entity_to_text(relation))
            for obj in objects:
                parts.append(entity_to_text(obj))

    return " ".join(parts)

def disambiguate(entityName, wikipediaArticle, knowledgeBase):
    """ Disambiguates the entity name based on the Wikipedia article.
    Returns None, or a yagoentity like <Paris_(mythology)>.
    """

    candidates = get_candidates(entityName, knowledgeBase)

    if not candidates:
        return None

    article_tokens = tokenize(wikipediaArticle)

    best_candidate = None
    best_score = -1

    for candidate in candidates:
        context = candidate_context(candidate, knowledgeBase)
        candidate_tokens = tokenize(context)

        overlap = article_tokens & candidate_tokens

        score = len(overlap)

        # Small bonus if words from the YAGO entity name appear in the article
        entity_name_tokens = tokenize(entity_to_text(candidate))
        score += 2 * len(article_tokens & entity_name_tokens)

        if score > best_score:
            best_score = score
            best_candidate = candidate

    # Precision is more important than recall in the evaluation.
    # If the model has almost no evidence, better return None.
    if best_score <= 0:
        return None

    return best_candidate

In [12]:
def run():
    """ Runs the disambiguation """
    kb=KnowledgeBase("yago-sample.tsv")
    print("Disambiguating entities...")
    with open("results.tsv", 'wt', encoding="utf-8") as output:
      start = time.time()
      for i, page in enumerate(wikipediaArticles("wikipedia-corpus.txt")):
        print("  Processing",page.title, i)
        result = disambiguate(page.name, page.content, kb)
        if result is not None:
          output.write(page.name +"\t" + page.entity_id + "\tyago:" + result[1:-1] + "\n")
    end = time.time()
    print("done")
    print("execution time: ", end - start)

In [13]:
run()
run_evaluation()

Loading yago-sample.tsv...done
Disambiguating entities...
  Processing Babilonia #1 0
  Processing Babilonia #2 1
  Processing Ashok Kumar #1 2
  Processing Ashok Kumar #2 3
  Processing Ashok Kumar #3 4
  Processing Ashok Kumar #4 5
  Processing Ashok Kumar #5 6
  Processing Ashok Kumar #6 7
  Processing Ashok Kumar #7 8
  Processing Ashok Kumar #8 9
  Processing Ashok Kumar #9 10
  Processing Mortal Kombat #1 11
  Processing Mortal Kombat #2 12
  Processing Mortal Kombat #3 13
  Processing Mortal Kombat #4 14
  Processing Mortal Kombat #5 15
  Processing Mortal Kombat #6 16
  Processing Kelston #1 17
  Processing Kelston #2 18
  Processing Kelston #3 19
  Processing Epirus #1 20
  Processing Epirus #2 21
  Processing Epirus #3 22
  Processing Marble Canyon #1 23
  Processing Marble Canyon #2 24
  Processing Marble Canyon #3 25
  Processing Marble Canyon #4 26
  Processing Foxfire #1 27
  Processing Foxfire #2 28
  Processing Foxfire #3 29
  Processing Foxfire #4 30
  Processing Foxfi